# Lib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)


# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Path to data
data_path = r"D:\Polythecninco di Milano\AFB_Lab\Data\Camihawke\Cleaned\ "
print("Data path:", data_path)

Data path: D:\Polythecninco di Milano\AFB_Lab\Data\Camihawke\Cleaned\ 


# Load Data

In [3]:
# Load all datasets
print("\n" + "=" * 80)
print("STAGE 1: DATA LOADING & OVERVIEW")
print("=" * 80)

# Camihawke - Facebook
fb_posts = pd.read_pickle(os.path.join(data_path.strip(), 'fb_posts_clean.pkl'))
fb_comments = pd.read_pickle(os.path.join(data_path.strip(), 'fb_comments_clean.pkl'))

# Camihawke - TikTok
tk_posts = pd.read_pickle(os.path.join(data_path.strip(), 'tk_posts_clean.pkl'))
tk_comments = pd.read_pickle(os.path.join(data_path.strip(), 'tk_comments_clean.pkl'))

# Camihawke - Instagram
ig_posts = pd.read_parquet(os.path.join(data_path.strip(), 'ig_posts_cleaned.parquet'))
ig_comments = pd.read_parquet(os.path.join(data_path.strip(), 'ig_comments_cleaned.parquet'))


# YouTube
yt_channels = pd.read_csv(os.path.join(data_path.strip(), 'channels_metadata.csv'))
yt_videos = pd.read_csv(os.path.join(data_path.strip(), 'videos_metadata.csv'))
yt_comments_1 = pd.read_parquet(os.path.join(data_path.strip(), 'YTcomments_1_cleaned.parquet'))
yt_comments_2 = pd.read_parquet(os.path.join(data_path.strip(), 'YTcomments_2_cleaned.parquet'))
yt_comments_3 = pd.read_parquet(os.path.join(data_path.strip(), 'YTcomments_3_cleaned.parquet'))
yt_comments_4 = pd.read_parquet(os.path.join(data_path.strip(), 'YTcomments_4_cleaned.parquet'))
yt_comments = pd.concat([yt_comments_1, yt_comments_2, yt_comments_3, yt_comments_4], ignore_index=True)

print("\n✓ All datasets loaded successfully\n")

# Data inventory
datasets = {
    'Facebook Posts': fb_posts,
    'Facebook Comments': fb_comments,
    'Instagram Posts': ig_posts,
    'Instagram Comments': ig_comments,
    'TikTok Posts': tk_posts,
    'TikTok Comments': tk_comments,
    'YouTube Channels': yt_channels,
    'YouTube Videos': yt_videos,
    'YouTube Comments': yt_comments
}

inventory = []
for name, df in datasets.items():
    inventory.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Memory (MB)': df.memory_usage(deep=True).sum() / 1024**2
    })

inventory_df = pd.DataFrame(inventory)
print("\nDATA INVENTORY:")
print(inventory_df.to_string(index=False))
print(f"\nTotal datasets: {len(datasets)}")


STAGE 1: DATA LOADING & OVERVIEW

✓ All datasets loaded successfully


DATA INVENTORY:
           Dataset    Rows  Columns  Memory (MB)
    Facebook Posts     390       29     0.465152
 Facebook Comments  394084        9   229.571702
   Instagram Posts    1540       18     1.614199
Instagram Comments  499752       13   299.831096
      TikTok Posts     335        8     0.087985
   TikTok Comments   19634       13     7.233101
  YouTube Channels      27       17     0.049412
    YouTube Videos   12839       67    77.949174
  YouTube Comments 3142109       14  2567.356218

Total datasets: 9


# Usernames


In [4]:
# Select username-related columns safely from ig_comments
wanted_cols = [c for c in ["from_id", "from_username"] if c in ig_comments.columns]
ig_usernames = ig_comments.loc[:, wanted_cols].copy()

ig_usernames.head()

,from_id,from_username
0,3482058005293604,giovannamirci
1,3103734239813346,alessia.daziano
2,908266745384508,guglielmoscilla
3,17841401147950242,camihawke
4,1709620023622902,alex_danci


In [5]:
# Select username-related columns safely from fb_comments
wanted_cols = [c for c in ["from_id", "from_name"] if c in fb_comments.columns]
fb_usernames = fb_comments.loc[:, wanted_cols].copy()

fb_usernames.head()

,from_id,from_name
0,35073676732216538,Giuseppe Floris Serra
1,26659197247010753,Fabio Flamigni
2,26693808403557118,Fabio Cicinelli
3,26022582060734103,Sisto Van Wilder Distratis
4,34655347520747705,Nicola Nik Corretto


In [6]:
# Select username-related columns safely from tk_comments
wanted_cols = [c for c in ["nickname", "uid"] if c in tk_comments.columns]
tk_usernames = tk_comments.loc[:, wanted_cols].copy()

tk_usernames.head()

,nickname,uid
0,Laura 🤪,7213726581705884678
1,Roby,6908466011760247810
2,💜🧿 valentain💜🧿,7046706435484353542
3,Alice,6799618304623821830
4,ele,7024167276347032581


In [7]:
import re

# Check for emoji in username columns across all three sources


def contains_emoji(text):
    """Check if text contains emoji"""
    if pd.isna(text):
        return False
    text = str(text)
    # Emoji Unicode ranges
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F9FF"  # Emoticons, symbols, pictographs
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F700-\U0001F77F"  # Alchemical Symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+"
    )
    return bool(emoji_pattern.search(text))

# Check Instagram usernames
ig_emoji_count = ig_usernames['from_username'].apply(contains_emoji).sum()
ig_total = ig_usernames['from_username'].notna().sum()
ig_emoji_pct = (ig_emoji_count / ig_total * 100) if ig_total > 0 else 0

# Check Facebook usernames
fb_emoji_count = fb_usernames['from_name'].apply(contains_emoji).sum()
fb_total = fb_usernames['from_name'].notna().sum()
fb_emoji_pct = (fb_emoji_count / fb_total * 100) if fb_total > 0 else 0

# Check TikTok usernames
tk_emoji_count = tk_usernames['nickname'].apply(contains_emoji).sum()
tk_total = tk_usernames['nickname'].notna().sum()
tk_emoji_pct = (tk_emoji_count / tk_total * 100) if tk_total > 0 else 0

# Summary table
emoji_summary = pd.DataFrame({
    'Source': ['Instagram', 'Facebook', 'TikTok'],
    'Usernames with Emoji': [ig_emoji_count, fb_emoji_count, tk_emoji_count],
    'Total Usernames': [ig_total, fb_total, tk_total],
    'Percentage (%)': [round(ig_emoji_pct, 2), round(fb_emoji_pct, 2), round(tk_emoji_pct, 2)]
})

print("\nEMOJI ANALYSIS IN USERNAMES:")
print(emoji_summary.to_string(index=False))


EMOJI ANALYSIS IN USERNAMES:
   Source  Usernames with Emoji  Total Usernames  Percentage (%)
Instagram                     0           499752            0.00
 Facebook                    48           394084            0.01
   TikTok                  2955            19634           15.05


In [8]:
# Reuse existing `re` and `pd` imports from previous cells

def remove_emojis(text):
    if pd.isna(text):
        return ""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub("", str(text))

# Apply to all three username datasets
ig_usernames["username_clean"] = (
    ig_usernames["from_username"].apply(remove_emojis).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

fb_usernames["username_clean"] = (
    fb_usernames["from_name"].apply(remove_emojis).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

tk_usernames["username_clean"] = (
    tk_usernames["nickname"].apply(remove_emojis).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

ig_usernames.head(), fb_usernames.head(), tk_usernames.head()

(             from_id    from_username   username_clean
 0   3482058005293604    giovannamirci    giovannamirci
 1   3103734239813346  alessia.daziano  alessia.daziano
 2    908266745384508  guglielmoscilla  guglielmoscilla
 3  17841401147950242        camihawke        camihawke
 4   1709620023622902       alex_danci       alex_danci,
              from_id                   from_name              username_clean
 0  35073676732216538       Giuseppe Floris Serra       giuseppe floris serra
 1  26659197247010753              Fabio Flamigni              fabio flamigni
 2  26693808403557118             Fabio Cicinelli             fabio cicinelli
 3  26022582060734103  Sisto Van Wilder Distratis  sisto van wilder distratis
 4  34655347520747705         Nicola Nik Corretto         nicola nik corretto,
          nickname                  uid username_clean
 0         Laura 🤪  7213726581705884678          laura
 1            Roby  6908466011760247810           roby
 2  💜🧿 valentain💜🧿  704670643

In [ ]:

try :
    from rapidfuzz import process, fuzz
except ImportError:
    print("rapidfuzz not found, installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rapidfuzz"])
    from rapidfuzz import process, fuzz
from multiprocessing import cpu_count, Pool


def _pick_cols(df, id_opts, clean_name_opts, raw_name_opts):
    id_col = next((c for c in id_opts if c in df.columns), None)
    clean_col = next((c for c in clean_name_opts if c in df.columns), None)
    raw_col = next((c for c in raw_name_opts if c in df.columns), None)
    return id_col, clean_col, raw_col


def build_users(df, source, id_opts, clean_name_opts, raw_name_opts):
    id_col, clean_col, raw_col = _pick_cols(df, id_opts, clean_name_opts, raw_name_opts)

    if clean_col is None and raw_col is None:
        raise ValueError(f"No username column found for source={source}")

    cols = [c for c in [id_col, clean_col, raw_col] if c is not None]
    df2 = df.loc[:, cols].copy()

    if id_col is not None:
        df2 = df2.rename(columns={id_col: "user_id"})
    else:
        df2["user_id"] = pd.NA

    name_col = clean_col if clean_col is not None else raw_col
    df2["username"] = df2[name_col].astype("string").fillna("").str.strip()
    df2["username_clean"] = (
        df2["username"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    df2["source"] = source

    df2 = df2[df2["username_clean"].ne("")]

    if df2["user_id"].notna().any():
        df2 = df2.drop_duplicates(subset=["user_id"], keep="first")
    df2 = df2.drop_duplicates(subset=["username_clean"], keep="first")

    return df2[["source", "user_id", "username", "username_clean"]].reset_index(drop=True)


ig_users = build_users(
    ig_usernames, "ig",
    id_opts=["from_id", "id", "user_id"],
    clean_name_opts=["username_clean", "from_username_clean", "clean_username"],
    raw_name_opts=["from_username", "username"]
)

fb_users = build_users(
    fb_usernames, "fb",
    id_opts=["from_id", "id", "user_id"],
    clean_name_opts=["username_clean", "from_name_clean", "clean_name"],
    raw_name_opts=["from_name", "name"]
)

tk_users = build_users(
    tk_usernames, "tk",
    id_opts=["uid", "id", "user_id"],
    clean_name_opts=["username_clean", "nickname_clean", "name_clean", "clean_username"],
    raw_name_opts=["nickname", "name", "username"]
)

print("Unique users (ig/fb/tk):", len(ig_users), len(fb_users), len(tk_users))

maps = {"ig": ig_users, "fb": fb_users, "tk": tk_users}


def _format_exact(df, a, b):
    out = df.rename(columns={
        "source_x": "source_a", "source_y": "source_b",
        "user_id_x": "user_id_a", "user_id_y": "user_id_b",
        "username_x": "username_a", "username_y": "username_b",
    }).copy()
    out["source_a"] = a
    out["source_b"] = b
    out["score"] = 100.0
    return out[["source_a", "user_id_a", "username_a", "source_b", "user_id_b", "username_b", "score"]]


# Multiprocessing worker for rapidfuzz matching
def _match_chunk(args):
    rows_chunk, B_names, B_rows, a, b, thresh = args
    out = []
    from rapidfuzz import process, fuzz
    for user_id, username, username_clean in rows_chunk:
        if not username_clean:
            continue
        m = process.extractOne(username_clean, B_names, scorer=fuzz.ratio, score_cutoff=thresh)
        if m is not None:
            b_row = B_rows[m[2]]
            out.append({
                "source_a": a, "user_id_a": user_id, "username_a": username,
                "source_b": b, "user_id_b": b_row["user_id"], "username_b": b_row["username"],
                "score": float(m[1]),
            })
    return out


def match_pair(A, B, a, b, thresh=85, parallel=True):
    """Match usernames between two sources using rapidfuzz with optional multiprocessing."""
    # Exact matches
    exact = A.merge(B, on="username_clean", how="inner", suffixes=("_x", "_y"))
    exact_out = _format_exact(exact, a, b) if len(exact) else pd.DataFrame(
        columns=["source_a", "user_id_a", "username_a", "source_b", "user_id_b", "username_b", "score"]
    )

    # Fuzzy matches on remaining usernames
    matched_names = set(exact["username_clean"]) if len(exact) else set()
    A2 = A[~A["username_clean"].isin(matched_names)].reset_index(drop=True)
    B2 = B[~B["username_clean"].isin(matched_names)].reset_index(drop=True)

    fuzzy_rows = []
    if len(A2) and len(B2):
        A_list = list(A2[["user_id", "username", "username_clean"]].itertuples(index=False, name=None))
        B_names = B2["username_clean"].tolist()
        B_rows = B2[["user_id", "username", "username_clean"]].to_dict(orient="records")

        if parallel and len(A_list) > 1000:
            # Parallel processing for large datasets
            workers = max(1, min(cpu_count() - 1, 8))
            n_chunks = workers * 4
            chunked = [A_list[i::n_chunks] for i in range(n_chunks) if any(True for _ in A_list[i::n_chunks])]
            args = [(chunk, B_names, B_rows, a, b, thresh) for chunk in chunked]
            with Pool(workers) as p:
                for res in p.imap_unordered(_match_chunk, args):
                    if res:
                        fuzzy_rows.extend(res)
        else:
            # Sequential processing for smaller datasets
            for user_id, username, username_clean in A_list:
                if not username_clean:
                    continue
                m = process.extractOne(username_clean, B_names, scorer=fuzz.ratio, score_cutoff=thresh)
                if m is not None:
                    b_row = B_rows[m[2]]
                    fuzzy_rows.append({
                        "source_a": a, "user_id_a": user_id, "username_a": username,
                        "source_b": b, "user_id_b": b_row["user_id"], "username_b": b_row["username"],
                        "score": float(m[1]),
                    })

    fuzzy_out = pd.DataFrame(
        fuzzy_rows,
        columns=["source_a", "user_id_a", "username_a", "source_b", "user_id_b", "username_b", "score"]
    )

    return pd.concat([exact_out, fuzzy_out], ignore_index=True)


# Run matching across all platform pairs
matches = []
for a, b in [("fb", "ig"), ("fb", "tk"), ("ig", "tk")]:
    matches.append(match_pair(maps[a], maps[b], a, b, thresh=85, parallel=True))

matches_df = (
    pd.concat(matches, ignore_index=True)
      .sort_values("score", ascending=False)
      .reset_index(drop=True)
)

print("Matches found:", len(matches_df))
matches_df.head(200)

Unique users (ig/fb/tk): 194513 187700 13452
